# GSimple Flux at Custom Detector Fiducial Volumes

Reads BNB gsimple flux files and computes the νμ flux that would pass through
different detector window shapes at different z positions.

**Physics setup:**
- Each gsimple entry gives a neutrino ray: origin `(vtxx, vtxy, vtxz)` (meters)
  at the upstream flux window (`vtxz ≈ −10 m`), plus 3-momentum `(px, py, pz)` (GeV).
- We project the ray to a target z-plane and check whether the projected `(x, y)` falls
  inside the detector window.
- z positions scanned: 0 m → 5 m (in the same coordinate system as vtxz).
- Two window definitions: `SBND` (full ±190 cm square) and `SBND_nohighyz` (y ≤ 100 cm).

In [ ]:
%load_ext autoreload
%autoreload 2

: 

In [ ]:
import os
import glob

import numpy as np
import matplotlib.pyplot as plt
import uproot
from tqdm import tqdm

: 

## Configuration

In [ ]:
GSIMPLE_DIR = (
    "/cvmfs/sbnd.osgstorage.org/pnfs/fnal.gov/usr/sbnd/persistent/stash/"
    "fluxFiles/bnb/BooNEtoGSimple/configK-v1/july2023/neutrinoMode/"
)

# Number of files to process (set to None to process all)
N_FILES = 100

# Target z positions to evaluate flux at (meters, same coords as vtxz)
Z_POSITIONS_M = [0.0, 1.0, 2.0, 3.0, 4.0, 5.0]

# Energy histogram binning
E_BINS = np.linspace(0, 3, 61)   # 0–3 GeV, 50 MeV bins

PDG_NUMU = 14

## Detector window definitions

Returns a boolean mask selecting neutrino projections that pass through the window.
Inputs `x` and `y` are in **cm**.

In [ ]:
DETECTORS = {
    "SBND": dict(xmin=-190., xmax=190., ymin=-190., ymax=190.),
    "SBND_nohighyz": dict(xmin=-190., xmax=190., ymin=-190., ymax=100.),
}


def in_window(x_cm, y_cm, det):
    """Boolean mask: does (x_cm, y_cm) fall inside the detector window?"""
    d = DETECTORS[det]
    return (
        (x_cm > d["xmin"]) & (x_cm < d["xmax"]) &
        (y_cm > d["ymin"]) & (y_cm < d["ymax"])
    )

## Explore file structure

Peek at one file to find the correct branch names.

In [ ]:
all_files = sorted(glob.glob(os.path.join(GSIMPLE_DIR, "*.root")))
print(f"Found {len(all_files)} files")

sample_file = all_files[0]
print("Sample file:", sample_file)

with uproot.open(sample_file) as f:
    print("\nKeys:", f.keys())

    print("\n--- flux tree branches ---")
    tree = f["flux"]   # gsimple trees are typically named 'flux'
    for k in tree.keys():
        print(" ", k)

    print("\n--- meta tree branches ---")
    meta_key = next((k for k in f.keys() if "meta" in k.lower()), None)
    if meta_key:
        meta = f[meta_key]
        for k in meta.keys():
            print(" ", k)
        # Print first entry of 'protons' (or similar) to confirm units
        pot_key = next((k for k in meta.keys() if "proton" in k.lower()), None)
        if pot_key:
            val = meta[pot_key].array(library="np")[0]
            print(f"\n  First entry of '{pot_key}': {val:.6g}")
    else:
        print("  (no meta tree found)")

## Load and project gsimple entries

For each file, load the νμ entries and project each ray to every target z plane.

In [ ]:
def _get_pot(f):
    """Read POT (protons on target) from the meta tree of an open uproot file."""
    meta_key = next((k for k in f.keys() if "meta" in k.lower()), None)
    if meta_key is None:
        raise RuntimeError("No meta tree found; cannot read POT.")
    meta = f[meta_key]
    pot_key = next((k for k in meta.keys() if "proton" in k.lower()), None)
    if pot_key is None:
        raise RuntimeError(f"No 'protons' branch found in meta tree; keys = {meta.keys()}")
    return float(meta[pot_key].array(library="np").sum())


def load_gsimple(filename):
    """Load νμ entries from a single gsimple ROOT file.

    Weights are divided by the file's simulated POT (read from the meta tree),
    so the returned 'wgt' array is per-POT.

    Returns a dict of 1-D arrays (already filtered to pdg == 14) plus 'pot'.
    Coordinates are converted to cm; energies stay in GeV.
    """
    with uproot.open(filename) as f:
        # ---- POT from meta tree ----
        pot = _get_pot(f)

        tree = f["flux"]

        # ---- Branch names: try struct-style 'entry/...' first, fall back to flat ----
        keys = tree.keys()
        if "entry/pdg" in keys:
            prefix = "entry/"
        elif "pdg" in keys:
            prefix = ""
        else:
            raise RuntimeError(f"Cannot find 'pdg' branch in {filename}; keys = {keys}")

        branches = [
            prefix + "pdg",
            prefix + "wgt",
            prefix + "vtxx",
            prefix + "vtxy",
            prefix + "vtxz",
            prefix + "px",
            prefix + "py",
            prefix + "pz",
            prefix + "E",
        ]
        arr = tree.arrays(branches, library="np")

    pdg  = arr[prefix + "pdg"]
    mask = pdg == PDG_NUMU

    return {
        "pot":  pot,
        "wgt":  arr[prefix + "wgt"][mask] / pot,    # scale to per-POT
        "vtxx": arr[prefix + "vtxx"][mask] * 100.,  # m → cm
        "vtxy": arr[prefix + "vtxy"][mask] * 100.,
        "vtxz": arr[prefix + "vtxz"][mask] * 100.,  # e.g. −1000 cm
        "px":   arr[prefix + "px"][mask],
        "py":   arr[prefix + "py"][mask],
        "pz":   arr[prefix + "pz"][mask],
        "E":    arr[prefix + "E"][mask],
    }

In [ ]:
files_to_process = all_files[:N_FILES] if N_FILES is not None else all_files
print(f"Processing {len(files_to_process)} file(s)…")

# Accumulate arrays across files
chunks = []
for fpath in tqdm(files_to_process):
    try:
        chunks.append(load_gsimple(fpath))
    except Exception as e:
        print(f"  [skip] {os.path.basename(fpath)}: {e}")

# Concatenate array fields; sum scalar POT separately
array_keys = [k for k in chunks[0] if k != "pot"]
data = {k: np.concatenate([c[k] for c in chunks]) for k in array_keys}
total_pot = sum(c["pot"] for c in chunks)

print(f"Total νμ entries : {len(data['E']):,}")
print(f"Total simulated POT: {total_pot:.4g}")

## Build flux histograms for each (z, detector) combination

In [ ]:
def project_to_z(data, z_target_m):
    """Project each neutrino ray to z = z_target_m (meters → cm).

    Returns (x_cm, y_cm) arrays at the target plane.
    """
    z_target_cm = z_target_m * 100.
    slope_x = data["px"] / data["pz"]   # dx/dz  (dimensionless)
    slope_y = data["py"] / data["pz"]
    dz = z_target_cm - data["vtxz"]      # distance along z-axis (cm)
    x_cm = data["vtxx"] + slope_x * dz
    y_cm = data["vtxy"] + slope_y * dz
    return x_cm, y_cm


# hists[det][z] = (counts, bin_edges)
hists = {det: {} for det in DETECTORS}

for z in Z_POSITIONS_M:
    x_cm, y_cm = project_to_z(data, z)
    for det in DETECTORS:
        mask = in_window(x_cm, y_cm, det)
        h, edges = np.histogram(data["E"][mask], bins=E_BINS, weights=data["wgt"][mask])
        hists[det][z] = (h, edges)

print("Done building histograms.")

## Plot: νμ flux spectrum at each z for each detector

In [ ]:
cmap = plt.cm.viridis
colors = [cmap(i / max(len(Z_POSITIONS_M) - 1, 1)) for i in range(len(Z_POSITIONS_M))]

fig, axes = plt.subplots(1, len(DETECTORS), figsize=(7 * len(DETECTORS), 4), sharey=False)

for ax, det in zip(np.atleast_1d(axes), DETECTORS):
    for color, z in zip(colors, Z_POSITIONS_M):
        h, edges = hists[det][z]
        ax.step(edges[:-1], h, where="post", color=color, linewidth=1.5, label=f"z = {z:.0f} m")

    ax.set_xlim(0, 3)
    ax.set_xlabel("Neutrino energy [GeV]")
    ax.set_ylabel(r"Flux [per POT per bin]")
    ax.set_title(rf"$\nu_\mu$ flux — {det}")
    ax.legend(fontsize=8, loc="upper right")

fig.tight_layout()
plt.show()

## Plot: ratio relative to z = 0 m

In [ ]:
fig, axes = plt.subplots(1, len(DETECTORS), figsize=(7 * len(DETECTORS), 4), sharey=True)

for ax, det in zip(np.atleast_1d(axes), DETECTORS):
    h_ref, edges = hists[det][Z_POSITIONS_M[0]]
    safe_ref = np.where(h_ref > 0, h_ref, np.nan)

    for color, z in zip(colors[1:], Z_POSITIONS_M[1:]):
        h, _ = hists[det][z]
        ratio = h / safe_ref
        ax.step(edges[:-1], ratio, where="post", color=color, linewidth=1.5, label=f"z = {z:.0f} m")

    ax.axhline(1, color="black", linewidth=0.8, linestyle="--")
    ax.set_xlim(0, 3)
    ax.set_ylim(0.5, 1.5)
    ax.set_xlabel("Neutrino energy [GeV]")
    ax.set_ylabel("Ratio to z = 0 m")
    ax.set_title(rf"Flux ratio — {det}")
    ax.legend(fontsize=8, loc="upper right")

fig.tight_layout()
plt.show()

## Plot: compare SBND vs SBND_nohighyz at each z

In [ ]:
n_cols = min(3, len(Z_POSITIONS_M))
n_rows = int(np.ceil(len(Z_POSITIONS_M) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 4 * n_rows), sharey=False)
axes = np.atleast_1d(axes).flatten()

det_colors = {"SBND": "C0", "SBND_nohighyz": "C1"}

for ax, z in zip(axes, Z_POSITIONS_M):
    for det, color in det_colors.items():
        h, edges = hists[det][z]
        ax.step(edges[:-1], h, where="post", color=color, linewidth=1.5, label=det)
    ax.set_xlim(0, 3)
    ax.set_xlabel("Neutrino energy [GeV]")
    ax.set_ylabel(r"Flux [per POT per bin]")
    ax.set_title(rf"z = {z:.0f} m")
    ax.legend(fontsize=8)

# Hide unused subplots
for ax in axes[len(Z_POSITIONS_M):]:
    ax.set_visible(False)

fig.suptitle(r"$\nu_\mu$ flux: SBND vs SBND\_nohighyz", y=1.01)
fig.tight_layout()
plt.show()

## Summary: integrated flux vs z

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))

for det, color in det_colors.items():
    integ = [hists[det][z][0].sum() for z in Z_POSITIONS_M]
    ax.plot(Z_POSITIONS_M, integ, marker="o", color=color, label=det)

ax.set_xlabel("z position [m]")
ax.set_ylabel(r"Integrated flux [per POT]")
ax.set_title(r"$\nu_\mu$ integrated flux vs z")
ax.legend()
fig.tight_layout()
plt.show()